In [2]:
!pip install wandb

In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PINN(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 3)
        )

    def forward(self, x):
        return self.net(x)

model = PINN().to(device)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shantanugupta2004 (shantanugupta2004-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
def sample_interior(n):
    x = torch.rand(n,1)
    y = torch.rand(n,1)
    pts = torch.cat([x,y], dim=1)
    return pts.to(device)

def sample_boundary(n):
    x = torch.rand(n,1)
    y = torch.rand(n,1)

    left   = torch.cat([torch.zeros_like(y), y], dim=1)
    right  = torch.cat([torch.ones_like(y), y], dim=1)
    bottom = torch.cat([x, torch.zeros_like(x)], dim=1)
    top    = torch.cat([x, torch.ones_like(x)], dim=1)

    return left.to(device), right.to(device), bottom.to(device), top.to(device)

In [5]:
def pde_residual(xy, Re):

    xy.requires_grad_(True)
    uvp = model(xy)

    u = uvp[:,0:1]
    v = uvp[:,1:2]
    p = uvp[:,2:3]

    u_x = torch.autograd.grad(u, xy, torch.ones_like(u), create_graph=True)[0][:,0:1]
    u_y = torch.autograd.grad(u, xy, torch.ones_like(u), create_graph=True)[0][:,1:2]

    v_x = torch.autograd.grad(v, xy, torch.ones_like(v), create_graph=True)[0][:,0:1]
    v_y = torch.autograd.grad(v, xy, torch.ones_like(v), create_graph=True)[0][:,1:2]

    p_x = torch.autograd.grad(p, xy, torch.ones_like(p), create_graph=True)[0][:,0:1]
    p_y = torch.autograd.grad(p, xy, torch.ones_like(p), create_graph=True)[0][:,1:2]

    u_xx = torch.autograd.grad(u_x, xy, torch.ones_like(u_x), create_graph=True)[0][:,0:1]
    u_yy = torch.autograd.grad(u_y, xy, torch.ones_like(u_y), create_graph=True)[0][:,1:2]

    v_xx = torch.autograd.grad(v_x, xy, torch.ones_like(v_x), create_graph=True)[0][:,0:1]
    v_yy = torch.autograd.grad(v_y, xy, torch.ones_like(v_y), create_graph=True)[0][:,1:2]

    # Continuity
    f_cont = u_x + v_y

    # Momentum
    f_u = u*u_x + v*u_y + p_x - (1/Re)*(u_xx + u_yy)
    f_v = u*v_x + v*v_y + p_y - (1/Re)*(v_xx + v_yy)

    return f_cont, f_u, f_v

In [6]:
wandb.init(
    project="SciML-PINN-LDC",
    name="PINN_Re50_LDC",
    config={
        "epochs": 5000,
        "lr": 1e-3,
        "Re": 50,
        "interior_points": 2000,
        "boundary_points": 500,
        "network": "3-layer MLP Tanh"
    }
)

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Re = 50
epochs = 5000

for epoch in range(epochs):

    # Sample points
    interior = sample_interior(2000)
    left, right, bottom, top = sample_boundary(500)

    # ----- PDE Residual -----
    f_cont, f_u, f_v = pde_residual(interior, Re)

    loss_cont = (f_cont**2).mean()
    loss_mom_u = (f_u**2).mean()
    loss_mom_v = (f_v**2).mean()

    loss_pde = loss_cont + loss_mom_u + loss_mom_v

    # ----- Boundary Conditions -----
    uvp_top = model(top)
    loss_top = ((uvp_top[:,0:1] - 1)**2).mean() + (uvp_top[:,1:2]**2).mean()

    uvp_left = model(left)
    uvp_right = model(right)
    uvp_bottom = model(bottom)

    loss_walls = (
        (uvp_left[:,0:2]**2).mean() +
        (uvp_right[:,0:2]**2).mean() +
        (uvp_bottom[:,0:2]**2).mean()
    )

    loss_bc = loss_top + loss_walls

    # ----- Total Loss -----
    loss = loss_pde + loss_bc

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # ----- Log Scalars -----
    wandb.log({
        "epoch": epoch,
        "total_loss": loss.item(),
        "pde_loss": loss_pde.item(),
        "continuity_loss": loss_cont.item(),
        "momentum_u_loss": loss_mom_u.item(),
        "momentum_v_loss": loss_mom_v.item(),
        "boundary_loss": loss_bc.item(),
    })

    # ----- Log Velocity Field -----
    if epoch % 1000 == 0:

        grid_size = 100
        x = torch.linspace(0,1,grid_size)
        y = torch.linspace(0,1,grid_size)
        grid_x, grid_y = torch.meshgrid(x,y, indexing='ij')
        grid = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1).to(device)

        with torch.no_grad():
            uvp = model(grid)
            u = uvp[:,0].reshape(grid_size,grid_size).cpu().numpy()
            v = uvp[:,1].reshape(grid_size,grid_size).cpu().numpy()

        fig, ax = plt.subplots(1,2, figsize=(10,4))
        im0 = ax[0].imshow(u)
        ax[0].set_title("u velocity")
        plt.colorbar(im0, ax=ax[0])

        im1 = ax[1].imshow(v)
        ax[1].set_title("v velocity")
        plt.colorbar(im1, ax=ax[1])

        wandb.log({"velocity_field": wandb.Image(fig)})
        plt.close()

    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Total Loss: {loss.item():.6f}")

wandb.finish()

Epoch 0 | Total Loss: 1.182848
Epoch 500 | Total Loss: 0.050336
Epoch 1000 | Total Loss: 0.048727
Epoch 1500 | Total Loss: 0.046466
Epoch 2000 | Total Loss: 0.043411
Epoch 2500 | Total Loss: 0.043760
Epoch 3000 | Total Loss: 0.034652
Epoch 3500 | Total Loss: 0.036494
Epoch 4000 | Total Loss: 0.033851
Epoch 4500 | Total Loss: 0.023853


boundary_loss,▇▇█▅▆▆██▅▆▅▄▆▅▅▅▅▅▃▅▄▄▃▄▃▄▄▂▃▂▂▁▂▁▃▁▂▂▁▁
continuity_loss,██▂▁▁▂▂▆█▃▃▆▂▂▁▂▂▄▃▂▁▃▂▂▁▂▁▂▂▃▂▁▃▆▂▂▃▂▄▂
epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
momentum_u_loss,▁▂▄▇█▃▄▄▄▅▄▇▄▃▃▃▄▆▅▃▂▄█▄▄▅▄▃▂▃█▂▃▂▂▂▃▂▂▂
momentum_v_loss,▁▁▅▅▆▄▅▅▅█▆▆▅▅▆▇▆▆▇▆▇▆▅▇▇▆▆▆▇▅▄▅▄▃▄▄▄▄▃▂
pde_loss,█▁▂▃█▃▃▄▄▃▄▃▃▃▄▅▄▅▃▃▃▃▃▇▂▄▃▃▃▃▄▂▂▂▂▂▃▄▃▁
total_loss,█▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁
boundary_loss,0.01806
continuity_loss,0.00059
epoch,4999
momentum_u_loss,0.00124
